# TransformerSubmission: Full H2 pipeline (local)

This notebook runs an end-to-end local pipeline for the Falcon Challenge H2 split using the repository's `HritikDecoder` (a JAX-based model):
- Validate local data (assumes datasets are already downloaded under `data/h2`).
- Optionally train a tiny checkpoint if none exists (fast, few epochs).
- Run minival evaluation to produce metrics and a submission payload (pickle) compatible with the evaluator.
- Additionally export a simple CSV view of predictions for convenience.

Notes
- This runs fully locally on minival data. Test-phase evaluation happens on the EvalAI server.
- If a model checkpoint exists at `local_data/hritik_h2/model.arrays.npz`, training is skipped.
- Set `JAX_PLATFORMS` to `cpu` or `cuda` if you have GPU support.


In [ ]:
# 1) Import Dependencies
import os, sys, json, math, time, random
from pathlib import Path
from typing import Dict, Any

import numpy as np
import pandas as pd

# Optional: torch/transformers (not required by pipeline, kept for outline completeness)
try:
    import torch  # noqa: F401
except Exception:
    torch = None
try:
    from transformers import AutoTokenizer, AutoModel  # noqa: F401
except Exception:
    AutoTokenizer = None
    AutoModel = None

from tqdm import tqdm

# Repo imports
from falcon_challenge.config import FalconConfig, FalconTask
from falcon_challenge.evaluator import FalconEvaluator
from decoder_demos.hritik_decoder import HritikDecoder, HritikConfig, TrainConfig, train_hritik


In [ ]:
# 2) Define Paths to Downloaded Artifacts
ROOT = Path.cwd()
DATA_ROOT = ROOT / 'data'
H2_DIR = DATA_ROOT / 'h2'
MINIVAL_DIR = H2_DIR / 'minival'
HELDIN_CALIB_DIR = H2_DIR / 'held_in_calib'

SAVE_ROOT = ROOT / 'local_data' / 'hritik_h2'
MODEL_BASE = SAVE_ROOT / 'model'  # used without suffix; arrays saved to .arrays.npz

# Submission-like payloads (pickles) created by evaluator
PREDICTION_PKL = ROOT / 'local_prediction.pkl'
GT_PKL = ROOT / 'local_gt.pkl'

print('ROOT          =', ROOT)
print('DATA_ROOT     =', DATA_ROOT)
print('MINIVAL_DIR   =', MINIVAL_DIR)
print('TRAINING_DIR  =', HELDIN_CALIB_DIR)
print('MODEL_BASE    =', MODEL_BASE)
print('PREDICTION_PKL=', PREDICTION_PKL)
print('GT_PKL        =', GT_PKL)


In [ ]:
# 3) Validate Files and Directories
assert H2_DIR.exists(), f"Missing {H2_DIR}. Please download H2 dataset under 'data/h2'."
assert MINIVAL_DIR.exists(), f"Missing {MINIVAL_DIR}."

minival_files = sorted(MINIVAL_DIR.glob('*.nwb'))
print(f"Found {len(minival_files)} minival files.")
for fn in minival_files[:5]:
    print(' -', fn.name)

# Set env vars for evaluator
os.environ['EVAL_DATA_PATH'] = str(DATA_ROOT)
os.environ['PREDICTION_PATH_LOCAL'] = str(PREDICTION_PKL)
os.environ['GT_PATH'] = str(GT_PKL)


In [ ]:
# 4) Load Run Configuration (with sane defaults)
run_cfg: Dict[str, Any] = {
    'phase': 'minival',
    'split': 'h2',
    'train': {
        'enable': True,          # if no checkpoint exists, train briefly
        'epochs': 1,             # keep fast
        'batch_size': 8,
        'lr': 1e-3,
        'max_len': 48,
    },
    'device': None,              # 'cpu' or 'cuda' to set JAX_PLATFORMS, or None to leave as-is
}
run_cfg


In [ ]:
# 5) Initialize Device and Reproducibility
if run_cfg.get('device'):
    os.environ['JAX_PLATFORMS'] = run_cfg['device']
    os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

seed = 0
random.seed(seed)
np.random.seed(seed)
if torch is not None:
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass

# Show JAX devices if available
try:
    import jax
    print('JAX devices:', jax.devices(), '| backend:', jax.default_backend())
except Exception as e:
    print('JAX not available or failed to import:', e)


In [ ]:
# 6) Load Tokenizer and Model (Local Checkpoint)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
ckpt_arrays = MODEL_BASE.with_suffix('.arrays.npz')

# If checkpoint is absent and training is enabled, do a quick train to create it
if not ckpt_arrays.exists() and run_cfg['train']['enable']:
    print('No checkpoint found. Kicking off a brief training run to create one...')
    cfg = HritikConfig()
    tcfg = TrainConfig(
        epochs=run_cfg['train']['epochs'],
        batch_size=run_cfg['train']['batch_size'],
        lr=run_cfg['train']['lr'],
        max_len=run_cfg['train']['max_len'],
    )
    train_hritik(
        training_dir=HELDIN_CALIB_DIR,
        heldout_calib_dir=None,
        save_path=MODEL_BASE,
        cfg=cfg,
        train_cfg=tcfg,
    )
else:
    print('Checkpoint exists, skipping training.')

# Instantiate decoder with the saved model
h2_cfg = FalconConfig(task=FalconTask.h2)
decoder = HritikDecoder(task_config=h2_cfg, model_path=str(MODEL_BASE), batch_size=1)
print('Decoder ready. Batch size =', decoder.batch_size)


### 7) Dataset and Preprocessing
For H2, the repository's evaluator handles NWB loading and trialization internally, so we don't implement a custom dataset here.


### 8) Create DataLoader
No explicit DataLoader is needed; `FalconEvaluator` constructs a PyTorch `DataLoader` under the hood with appropriate collate logic for H2.


In [ ]:
# 9) Inference Loop (Evaluator-driven)
start_time = time.time()
evaluator = FalconEvaluator(eval_remote=False, split='h2', verbose=False, dataloader_workers=0)
metrics = evaluator.evaluate(decoder=decoder, phase=run_cfg['phase'])
elapsed = time.time() - start_time
print('Metrics:', metrics)
print(f'Elapsed: {elapsed:.2f}s')


In [ ]:
# 10) Post-process Predictions -> flat DataFrame view
import pickle
from collections import defaultdict

with open(PREDICTION_PKL, 'rb') as f:
    pred_payload = pickle.load(f)

# Structure: { 'h2': { hash: [ [trial strings...], ... ], 'normalized_latency': float } }
assert 'h2' in pred_payload, f'Unexpected payload keys: {list(pred_payload.keys())}'
h2_pred = pred_payload['h2']
latency = h2_pred.pop('normalized_latency', None)

rows = []
for data_hash, sess_pred in h2_pred.items():
    # sess_pred is list with batch dim 1: [ [str per trial], ... ]
    # We flatten to (data_hash, trial_idx, prediction)
    if isinstance(sess_pred, list) and len(sess_pred) > 0 and isinstance(sess_pred[0], list):
        sess_pred = sess_pred[0]
    for t_idx, pred_str in enumerate(sess_pred):
        rows.append({'data_hash': data_hash, 'trial_idx': t_idx, 'prediction': pred_str})

pred_df = pd.DataFrame(rows)
print('Rows:', len(pred_df), '| unique files:', pred_df['data_hash'].nunique(), '| latency:', latency)
pred_df.head()


In [ ]:
# 10b) Robust flattening of H2 predictions (handles nested list structure)
# Reuse pred_payload if present; otherwise reload
try:
    h2_pred  # type: ignore[name-defined]
except NameError:
    import pickle
    with open(PREDICTION_PKL, 'rb') as f:
        pred_payload = pickle.load(f)
    assert 'h2' in pred_payload, f'Unexpected payload keys: {list(pred_payload.keys())}'
    h2_pred = pred_payload['h2']

latency = h2_pred.pop('normalized_latency', None)

rows = []
for data_hash, sess_pred in h2_pred.items():
    # sess_pred can be a list of trial lists (appended per dataloader chunk)
    flat_trials = []
    if isinstance(sess_pred, list):
        for chunk in sess_pred:
            if isinstance(chunk, list):
                # chunk may already be a list of strings or a batched list [ [strs...] ]
                if len(chunk) > 0 and isinstance(chunk[0], list):
                    for inner in chunk:
                        flat_trials.extend(inner)
                else:
                    flat_trials.extend(chunk)
            else:
                flat_trials.append(chunk)
    else:
        flat_trials = [sess_pred]

    for t_idx, pred_str in enumerate(flat_trials):
        rows.append({'data_hash': data_hash, 'trial_idx': t_idx, 'prediction': pred_str})

pred_df = pd.DataFrame(rows)
print('Rows:', len(pred_df), '| unique files:', pred_df['data_hash'].nunique(), '| latency:', latency)
pred_df.head()

In [ ]:
# 11) Build and Save Submission CSV (for convenience)
SUBMISSION_CSV = ROOT / 'submission.csv'
pred_df.to_csv(SUBMISSION_CSV, index=False)
print('Wrote CSV ->', SUBMISSION_CSV.resolve())

# Note: For EvalAI submissions, this repo uses the pickle payload already saved at PREDICTION_PKL.


In [ ]:
# 12) Validate Submission and Runtime Summary
assert pred_df.notnull().all().all(), 'Nulls found in predictions DataFrame.'
print('CSV size (bytes):', SUBMISSION_CSV.stat().st_size)

try:
    import jax
    devs = jax.devices()
    print('Ran on JAX backend:', jax.default_backend(), '| devices:', devs)
except Exception:
    pass

print('Done.')
